In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),                        
    transforms.Normalize((0.1307,), (0.3081,))      
])

train_dataset = torchvision.datasets.MNIST(
    root='./data',      
    train=True,           
    transform=transform,   
    download=True          
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,       
    num_workers=2
)

# 验证：查看一个 batch 的形状
images, labels = next(iter(train_loader))
print(f"图像 batch 形状: {images.shape}")   
print(f"标签 batch 形状: {labels.shape}")    # [64]
print(f"像素值范围: [{images.min():.3f}, {images.max():.3f}]")   # 归一化后约为 [-0.42, 2.82]

# 可视化前 8 张图
fig, axes = plt.subplots(1, 8, figsize=(12, 3))
for i in range(8):
    img = images[i].squeeze(0)  
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"标签: {labels[i].item()}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import model
import torch.optim as optim
import numpy as np

T = 1000
Model=model.Diffusion(T)
optimizer = optim.Adam(Model.parameters(), lr=0.001, weight_decay=1e-4)

# train

num_epochs = 10
for epochs in range(num_epochs):
    for images, labels in train_loader:
        t=np.random.randint(1,T,size=64)
        t=torch.tensor(t)
        Loss=Model(images,t)
        
        optimizer.zero_grad()
        Loss.backward()
        optimizer.step()

In [ ]:
# eval
X=torch.randn([64,1,32,32])
for i in range(T-1,0,-1):
    t=torch.full(64,i)
    X=Model.eval(X,t)

fig, axes = plt.subplots(1, 8, figsize=(12, 3))
for i in range(8):
    img = X[i].squeeze(0)  
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
plt.tight_layout()
plt.show()